# ChurnGuard — exploratory data analysis

This notebook documents the dataset contract and the assumptions behind the
feature engineering. It is intentionally read-only: production logic lives in
`src/churnguard/` and must not be duplicated here.

Run `uv run dvc repro` first so the processed Parquet files exist.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from churnguard.config import get_settings
from churnguard.constants import NUMERIC_FEATURES, CATEGORICAL_FEATURES

pd.set_option("display.max_columns", 50)
settings = get_settings()
train = pd.read_parquet(settings.train_path)
test = pd.read_parquet(settings.test_path)
train.shape, test.shape

## 1. Class balance

The dataset is imbalanced (~26.5% churn). This is why training uses
`scale_pos_weight` and why the decision threshold is tuned rather than fixed
at 0.5.

In [ ]:
rate = train["Churn"].mean()
print(f"churn rate: {rate:.1%}")
train["Churn"].value_counts(normalize=True).plot(kind="bar", title="Churn balance")
plt.show()

## 2. Contract check

All columns below are validated by Pandera before training. Unknown categories
are rejected at the data gate, while the serving pipeline additionally tolerates
unknown categories via `OneHotEncoder(handle_unknown="ignore")`.

In [ ]:
print("numeric:", NUMERIC_FEATURES)
print("categorical:", CATEGORICAL_FEATURES)
train.dtypes.to_frame("dtype")

## 3. Churn drivers

Contract type and tenure dominate. `Month-to-month` customers churn far more
than one/two-year contracts, which is why `is_month_to_month` is engineered as
an explicit feature.

In [ ]:
for column in ["Contract", "InternetService", "PaymentMethod"]:
    train.groupby(column, observed=True)["Churn"].mean().sort_values().plot(
        kind="barh", title=f"Churn rate by {column}"
    )
    plt.show()

In [ ]:
train.groupby(pd.cut(train["tenure"], bins=[0, 6, 12, 24, 48, 72]), observed=True)["Churn"].agg(
    ["mean", "count"]
).rename(columns={"mean": "churn_rate"})

## 4. Missing values

`TotalCharges` arrives as a string with blank values for zero-tenure customers.
Cleaning coerces to numeric and fills with `0.0`; the numeric imputer remains as
a safety net for future upstream drift.

In [ ]:
train.isna().sum().sort_values(ascending=False).head(10)

## 5. Distribution shift guardrail

The train split is also written as the drift **reference**. Any future batch is
compared against it with Evidently (per-column tests) and the drift share is
exported to Prometheus.

Next: open `reports/drift/latest.html` after running
`uv run python -m churnguard.cli simulate-drift && uv run python -m churnguard.cli drift`.